Copyright Hewlett Packard Enterprise Development LP.

# ORNL Telemetry Data Across All Caps using Arkouda

This is an attempt to transliterate the code in Data_across_All_Caps.ipynb that uses pandas into arkouda.

In [ ]:
import arkouda as ak

In [ ]:
ak.connect("node-name")

In [ ]:
dir = "/lus/scratch/khandeka/dev/arkouda-telemetry-analysis/parquet-traces-for-LSMS-application-16x/"

power_cap200 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_200_16_events*"
power_cap400 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_400_16_events*"
power_cap300 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_300_16_events*"
power_cap500 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_500_16_events*"
power_cap600 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_600_16_events*"
power_cap700 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_700_16_events*"
power_cap800 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_800_16_events*"
power_cap900 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_900_16_events*"
power_cap1000 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_1000_16_events*"

# List of power cap files
power_caps = [power_cap200, power_cap300, power_cap400, power_cap500, power_cap600, power_cap700, power_cap800, power_cap900, power_cap1000]
# power_caps = [power_cap1000]

power_cap_names = [name.split('_')[-3] for name in power_caps[::-1]]
print(power_cap_names)  # Output: ['800', '900', '1000']

In [ ]:
def read_parquet_and_drop_columns(file_path):
    ak_df = ak.read_parquet(file_path)
    # Drop the columns we don't need
    keys_to_keep = ['time', 'metric', 'values', 'region', 'location', 'event']
    ak_df = {key: ak_df[key] for key in keys_to_keep}
    ak_df = ak.DataFrame(ak_df)
    return ak_df

In [ ]:
def drop_columns_except(ak_df, columns_to_keep):
    # Drop the columns we don't need
    for col in ak_df.columns:
        if col not in columns_to_keep:
            ak_df = ak_df.drop(col, axis='columns')
    return ak_df

In [ ]:
def preprocess_power_data(df):
    df = df[df["values"] != "nan"]
    df['values'] = df['values'].strip('[]').astype(ak.float64) / 1000 # convert miliwatts to watts
    return df

def find_leave_events(df):
    leave_events = df[df["event"] == "Leave"]
    # leave_events = leave_events.sort_values("time") # They are already sorted by time
    return leave_events

In [ ]:
import arkouda.array_api as xp
import time

def process_power_data_approximation(df_events, df_power):
    # We know that both dataframes are sorted by time already

    slice_start = time.time()
    gpudf_time = df_events["time"]
    powerdf_time = df_power["time"]
    powerdf_power = df_power["values"]
    slice_time = time.time() - slice_start
    print(f"Time to slice dataframes: {slice_time:.4f} seconds")

    if_start = time.time()
    # Make sure that the powerdf has an entry at time 0
    # This is important because we want to align the power data with the event data
    # searchsorted-1 will return a bad index if we don't do this
    # see if we can avoid the concat by always leaving an entry at time 0
    if powerdf_time[0] > 0:
        first_power_value = powerdf_power[0]  # Get the first power value
        # Create a new entry at time 0 with the first power value
        # We need to use the same type as the original powerdf_power
        # Create new arrays with the first entry
        powerdf_time = ak.concatenate((ak.array([0], dtype=powerdf_time.dtype), powerdf_time))
        powerdf_power = ak.concatenate((ak.array([first_power_value], dtype=powerdf_power.dtype), powerdf_power))
    if_time = time.time() - if_start
    print(f"Time to check and insert first power value: {if_time:.4f} seconds")

    # Find indices in powerdf_time where gpudf_time should align
    search_start = time.time()
    indices = ak.searchsorted(powerdf_time, gpudf_time, side='right', x2_sorted=True)
    search_time = time.time() - search_start
    print(f"Time for searchsorted: {search_time:.4f} seconds")

    # print(indices)

    # Ensure indices are valid (i.e., not negative)
    # indices = ak.where(indices < 0, 0, indices)

    # Retrieve the corresponding power values
    aligned_start = time.time()
    indices -= 1  # Adjust indices to get the correct power value
    aligned_power = powerdf_power[indices]
    aligned_time = time.time() - aligned_start
    print(f"Time to get aligned power data: {aligned_time:.4f} seconds")

    # Sanity checks
    if ak.isnan(aligned_power).any():
        raise ValueError("NaN values found in the aligned power data. Check your indices or input data.")
    if aligned_power.size == 0:
        raise ValueError("Aligned power data is empty. Check your input data or indices.")
    if aligned_power.size != gpudf_time.size:
        raise ValueError("Aligned power data size does not match event data size. Check your indices or input data.")

    return aligned_power

#### Note on the difference in approach when using Arkouda
Iterating directly over a DataFrame with `for x in df` is not recommended. Doing so is discouraged because it requires transferring all array data from the arkouda server to the Python client since there is almost always a more array-oriented way to express an iterator-based computation.

In [ ]:
import arkouda.array_api as xp
import numpy as np
import time as time

def getInclusiveEnergy(df_events):
    setup_time = time.time()
    df_power = df_events[df_events["metric"] == "MetricInstance [21]: 'metric_class': MetricClass [20], 'recorder': Location [47244640256] '', 'metric_scope': MetricScope.SYSTEM_TREE_NODE, 'scope': SystemTreeNode [1] 'wombat35'"].copy()

    regions = df_events[df_events["region"] != "nan"]  # Filter out rows where region is NaN
    df_gpuevents = regions[regions["location"] == "CUDA[0:7]"].copy()
    setup_time = time.time() - setup_time


    region_slice_time = 0
    enter_leave_time = 0
    duration_time = 0
    energy_mask_time = 0
    energy_df_slices_time = 0
    trapz_vectorized_time = 0
    misc_time = 0
    inner_loop_time = 0

    start_time = time.time()
    df_power = preprocess_power_data(df_power)
    preprocess_power_data_time = time.time() - start_time
    approx_start = time.time()
    approximated_power = process_power_data_approximation(df_events, df_power)
    approx_time = time.time() - approx_start
    # df_gpuevents = drop_columns_except(df_gpuevents, ['time', 'event', 'region'])

    other_preprocess_time = time.time()
    # Print description of the dataframe types
    # print(df_gpuevents.dtypes)
    region_col  = df_gpuevents['region']
    # print(type(region_col))
    event_col  = df_gpuevents['event']
    # print(type(event_col))
    time_col  = df_gpuevents['time']
    # print(type(time_col))
    regions = ak.unique(region_col)
    # print(regions)

    # Converting to categorical data type for "better performance
    # The performance is a couple of seconds slower per file, so we don't do this for now
    # region_col = ak.Categorical(region_col)
    # print(type(region_col))
    # event_col = ak.Categorical(event_col)
    # print(type(event_col))
    # print(event_col.categories)

    # Storing the total energy and time for each region in pdarrays
    region_times = ak.zeros(len(regions), dtype=ak.float64)
    region_energies = ak.zeros(len(regions), dtype=ak.float64)
    print(f"Number of regions: {len(regions)}")
    regions_ndarray = regions.to_ndarray()
    other_preprocess_time = time.time() - other_preprocess_time
    total_preprocess_time = preprocess_power_data_time + approx_time + other_preprocess_time
    # The data has 36 (?) or so unique regions, so we can loop over them like this without it being terrible
    # We could also sort the data by region and then just slice into the df instead of having to make multiple linear searches
    # But that's a potential future optimization
    for i, region in enumerate(regions_ndarray):
        inner_loop_start = time.time()
        region_slice_start = time.time()
        df_r_mask = region_col == region
        # print(f"Region: {region}, Number of data points: {df_r}")
        # if df_r_mask.any() == False:
        #     print(f"Region: {region}, has no data points, skipping")
        #     continue

        region_slice_time += time.time() - region_slice_start
        enter_leave_start = time.time()
        # I would love to use .apply() here to do this in parallel, but I don't think arkouda supports that yet
        enter_events_mask = df_r_mask & (event_col == 'Enter')
        leave_events_mask = df_r_mask & (event_col == 'Leave')
        enter_leave_time += time.time() - enter_leave_start

        cast_start = time.time()
        leave_times = time_col[leave_events_mask]
        # leave_regions = leave_events['region'] # This is not needed since region will be the same in this loop

        enter_times = time_col[enter_events_mask]
        # enter_regions = enter_events["region"] # This is not needed since region will be the same in this loop
        misc_time += time.time() - cast_start
        """"
        enter time: 35 (k1)
        enter time: 35 (k2)

        leave times10 20 30 35 40 40 50
                               k2 k1
        """
        dur_start = time.time()
        leave_indices = xp.searchsorted(xp.asarray(leave_times), xp.asarray(enter_times), side="right", x2_sorted=True)._array

        valid_indices = (leave_indices < len(leave_times)) #& (leave_regions[leave_indices] == enter_regions)

        valid_enter_times = enter_times[valid_indices]
        valid_leave_times = leave_times[valid_indices]
        durations = valid_leave_times - valid_enter_times
        total_region_time = ak.sum(durations)
        region_times[i] = total_region_time  # Accumulate time for each region

        duration_time += time.time() - dur_start
        # print(f"Runtime for durations: {time.time() - dur_start:.4f} seconds")


        # Vectorized energy calculation
        energy_mask_start = time.time()
        # Use sorted property to efficiently compute the mask
        enter_idx = xp.searchsorted(xp.asarray(df_events['time']), xp.asarray(valid_enter_times), side='left', x2_sorted=True)._array
        leave_idx = xp.searchsorted(xp.asarray(df_events['time']), xp.asarray(valid_leave_times), side='right', x2_sorted=True)._array
        # print(enter_idx)
        # print(leave_idx)

        # Create a delta array to track changes
        delta = ak.zeros(len(df_events['time']) + 1, dtype=int)
        delta[enter_idx] += 1
        delta[leave_idx] -= 1

        # Compute the cumulative sum to get the final mask
        energy_mask = ak.cumsum(delta[:-1]) > 0

        # print(mask)

        energy_mask_time += time.time() - energy_mask_start
        # print(f"Runtime for energy mask: {time.time() - energy_mask_start:.4f} seconds")

        slice_time = time.time()

        energy_values = ak.where(energy_mask, approximated_power, 0) # Use the mask to get the power values during the event, power=0 when kernel not running

        energy_df_slices_time += time.time() - slice_time
        # print(f"Runtime for energy df slices: {time.time() - slice_time:.4f} seconds")

        trapz_start = time.time()

        total_region_energy = xp.trapz(xp.asarray(energy_values), xp.asarray(df_events['time']))._array

        trapz_vectorized_time += time.time() - trapz_start
        # print(f"Runtime for trapz vectorized: {time.time() - trapz_start:.4f} seconds")
        misc_start = time.time()
        region_energies[i] = total_region_energy  # Accumulate energy for each region
        misc_time += time.time() - misc_start
        # print(f"Runtime for misc: {time.time() - misc_start:.4f} seconds")
        # print("Region Energy: ", total_region_energy)
        # print("Region Time: ", total_region_time)
        inner_loop_time += time.time() - inner_loop_start
        # print(f"Runtime for inner loop: {time.time() - inner_loop_start:.4f} seconds")

    print_time = time.time()
    # print("Region Energies: ", region_energies)
    # print("Region Times: ", region_times)

    energy_data = {
        'region': regions,
        'energy': region_energies,
        'duration': region_times
    }
    print(energy_data)
    df_energies = ak.DataFrame(energy_data)
    df_energies = df_energies.sort_values('energy', ascending=False)
    df_energies = df_energies[df_energies['energy'] > 1.0]
    print_time = time.time() - print_time

    end_time = time.time()
    runtime = end_time - start_time
    print(f"Setup Time: {setup_time:.2f} seconds")
    print(f"Runtime: {runtime:.2f} seconds")
    print(f"Preprocess Time: {total_preprocess_time:.2f} seconds")
    print(f"  Preprocess Power data Time: {preprocess_power_data_time:.2f} seconds")
    print(f"  Approximation Power data Time: {approx_time:.2f} seconds")
    print(f"  Other Preprocess Time: {other_preprocess_time:.2f} seconds")
    print(f"Inner Loop Time: {inner_loop_time:.2f} seconds")
    print(f"  Region Slice Time: {region_slice_time:.2f} seconds")
    print(f"  Enter Leave Time: {enter_leave_time:.2f} seconds")
    print(f"  Durations Time: {duration_time:.2f} seconds")
    print(f"  Energy Mask Time: {energy_mask_time:.2f} seconds")
    print(f"  Energy DF Slices Time: {energy_df_slices_time:.2f} seconds")
    print(f"  Trapz Vectorized Time: {trapz_vectorized_time:.2f} seconds")
    print(f"  Misc Time: {misc_time:.2f} seconds")
    print(f"Print Time: {print_time:.2f} seconds")

    # There might be a bug here
    df_energies_pandas = df_energies.to_pandas(retain_index=True)
    # print(df_energies_pandas)

    # Output the regions and their total energies
    print("\n=== Total Energy per Region ===")
    for region_name, energy, total_time in zip(df_energies_pandas['region'], df_energies_pandas['energy'], df_energies_pandas['duration']):
        print(f"Region: {region_name}, Total Energy: {energy:.2f} J, Total Time: {total_time:.2f} s")

    return df_energies_pandas.head(8), total_preprocess_time, inner_loop_time # I just do to_pandas here to minimize the amount of data that needs to be transferred back to the client


In [ ]:
def get_avg_power(df_energies):
    # Add the avg power column to the same DataFrame
    # Nice for a demo, but you would just do the division here instead of using assign
    # df_energies.assign(avg_power=lambda x: (x['energy']*x['duration']))

    return {region: avg_power for region, avg_power in zip(df_energies['region'].values, df_energies['energy'].values/df_energies['duration'].values)}


In [ ]:
def get_speedup_perf_drop(df_energies, baseline):
    speedup_energy = {}
    performance_drop = {}
    energy_saved = {}

    for _, row in df_energies.iterrows():
        region = row['region']
        energy = row['energy']
        runtime = row['duration']
        baseline_runtime = baseline.loc[baseline['region'] == region, 'duration'].values[0]
        baseline_energy = baseline.loc[baseline['region'] == region, 'energy'].values[0]

        if baseline_runtime and baseline_energy and runtime and energy:
            speedup = (baseline_runtime * baseline_energy) / (runtime * energy)
            energy_saved[region] = 100 * (energy - baseline_energy) / baseline_energy if energy != 0 else None
            performance_drop[region] = 100 * (runtime - baseline_runtime) / baseline_runtime if runtime != 0 else None
        else:
            speedup = None
            energy_saved[region] = None
            performance_drop[region] = None

        speedup_energy[region] = speedup

    return energy_saved, performance_drop, speedup_energy


In [ ]:
avg_power_data = {}
energy_saved_data = {}
performance_drop_data = {}
speedup_energy_data = {}

read_time = 0
initial_calc_time = 0
energy_preprocess_time = 0
energy_loop_time = 0
energy_calc_time = 0
other_calc_time = 0

for cap in power_caps[::-1]:
    print(f"Processing {cap}...")
    read_start_time = time.time()
    df_events = read_parquet_and_drop_columns(dir+cap)
    read_time += time.time() - read_start_time

    init_calc_start_time = time.time()
    df_events["time"] = df_events["time"].astype(ak.int64) # Convert time to seconds
    df_events["time"] = (df_events['time'] - df_events['time'][0]) / 1000000000
    initial_calc_time += time.time() - init_calc_start_time

    energy_calc_start_time = time.time()
    df_energies, preprocess_time, loop_time = getInclusiveEnergy(df_events)
    energy_calc_time += time.time() - energy_calc_start_time
    energy_preprocess_time += preprocess_time
    energy_loop_time += loop_time

    other_calc_start_time = time.time()

    if cap == power_cap1000:
        baseline = df_energies

    # use arrays and dataframes
    avg_power = get_avg_power(df_energies)
    energy_saved, performance_drop, speedup_energy  = get_speedup_perf_drop(df_energies, baseline)

    # Store metrics for each region across power caps
    for region in df_energies['region'].values:
        if region not in avg_power_data:
            avg_power_data[region] = []
            energy_saved_data[region] = []
            performance_drop_data[region] = []
            speedup_energy_data[region] = []

        avg_power_data[region].append(avg_power.get(region, None))
        energy_saved_data[region].append(energy_saved.get(region, None))
        performance_drop_data[region].append(performance_drop.get(region, None))
        speedup_energy_data[region].append(speedup_energy.get(region, None))
    other_calc_time += time.time() - other_calc_start_time

print(f"Read Time: {read_time:.2f} seconds")
print(f"Initial Calculation Time: {initial_calc_time:.2f} seconds")
print(f"Energy Calculation Time: {energy_calc_time:.2f} seconds")
print(f"  Energy Preprocess Time: {energy_preprocess_time:.2f} seconds")
print(f"  Energy Loop Time: {energy_loop_time:.2f} seconds")
print(f"Other Calculation Time: {other_calc_time:.2f} seconds")

The performance here is as follows:

1 locale
```
Setup Time: 7.95 seconds
Runtime: 30.34 seconds
Preprocess Time: 1.73 seconds
Inner Loop Time: 28.22 seconds
  Durations Time: 9.20 seconds
  Energy Mask Time: 4.64 seconds
  Energy DF Slices Time: 1.99 seconds
  Trapz Vectorized Time: 12.32 seconds
  Misc Time: 0.07 seconds
Print Time: 0.39 seconds
```

2 locale

```
Setup Time: 7.05 seconds
Runtime: 143.46 seconds
Preprocess Time: 13.08 seconds
Inner Loop Time: 129.35 seconds
  Durations Time: 17.49 seconds
  Energy Mask Time: 6.31 seconds
  Energy DF Slices Time: 1.75 seconds
  Trapz Vectorized Time: 103.67 seconds
  Misc Time: 0.13 seconds
Print Time: 1.02 seconds
```

16 locale

```
Setup Time: 2.43 seconds
Runtime: 142.80 seconds
Preprocess Time: 10.43 seconds
Inner Loop Time: 131.24 seconds
  Durations Time: 22.09 seconds
  Energy Mask Time: 4.46 seconds
  Energy DF Slices Time: 0.50 seconds
  Trapz Vectorized Time: 104.04 seconds
  Misc Time: 0.14 seconds
Print Time: 1.13 seconds
```

After improving diff, 16 locale

```
Setup Time: 2.56 seconds
Runtime: 41.16 seconds
Preprocess Time: 9.26 seconds
Inner Loop Time: 30.79 seconds
  Durations Time: 22.01 seconds
  Energy Mask Time: 4.42 seconds
  Energy DF Slices Time: 0.50 seconds
  Trapz Vectorized Time: 3.72 seconds
  Misc Time: 0.13 seconds
Print Time: 1.11 seconds
```

For all files combined:
```
Read Time: 120.66 seconds
Energy Calculation Time: 417.84 seconds
Other Calculation Time: 1.61 seconds
```


On my mac, when working with non-distributed install of Arkouda:

```
Setup Time: 3.17 seconds
Runtime: 8.56 seconds
Preprocess Time: 0.35 seconds
Inner Loop Time: 8.16 seconds
  Durations Time: 2.82 seconds
  Energy Mask Time: 2.00 seconds
  Energy DF Slices Time: 0.23 seconds
  Trapz Vectorized Time: 3.09 seconds
  Misc Time: 0.03 seconds
Print Time: 0.05 seconds
```

```
Read Time: 84.65 seconds
Initial Calculation Time: 9.52 seconds
Energy Calculation Time: 102.76 seconds
Other Calculation Time: 0.02 seconds
```

On my mac, but distributed install, running single node

```
Setup Time: 12.46 seconds
Runtime: 17.72 seconds
Preprocess Time: 0.69 seconds
Inner Loop Time: 16.98 seconds
  Durations Time: 3.96 seconds
  Energy Mask Time: 1.89 seconds
  Energy DF Slices Time: 0.23 seconds
  Trapz Vectorized Time: 10.86 seconds
  Misc Time: 0.02 seconds
Print Time: 0.05 seconds
```


```
Read Time: 84.68 seconds
Initial Calculation Time: 10.92 seconds
Energy Calculation Time: 276.46 seconds
Other Calculation Time: 0.03 seconds
```

On hotlum again doing scaling.

16 nodes Base data:
```
Read Time: 124.73 seconds
Initial Calculation Time: 0.61 seconds
Energy Calculation Time: 327.17 seconds
Other Calculation Time: 0.06 seconds
```



Plotting code would be the same

In [ ]:
import matplotlib.pyplot as plt

# Plotting function
def plot_metric_across_power_caps(data, title, ylabel):
    plt.figure(figsize=(8, 5))
    for region, values in data.items():
        if region == "COMPUTE IDLE":
            continue
        # Shorten the region name for the legend
        short_region = region[:15] + "..." if len(region) > 15 else region
        plt.plot(power_cap_names, values, marker='o', label=short_region)
    plt.xlabel("Power Cap (W)")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend(loc="best")
    plt.grid(True)
    plt.show()

# Plot each metric
plot_metric_across_power_caps(avg_power_data, "Average Power across Power Caps", "Average Power (W)")
plot_metric_across_power_caps(energy_saved_data, "Energy Saved across Power Caps", "Energy Saved (%)")
plot_metric_across_power_caps(performance_drop_data, "Performance Drop across Power Caps", "Performance Drop (%)")
plot_metric_across_power_caps(speedup_energy_data, "Speedup across Power Caps", "Speedup")